In [ ]:
import pandas as pd; import numpy as np ; import matplotlib.pyplot as plt ; import seaborn as sns;
from numpy import argmax;
import scipy.stats as st;

from sklearn.datasets import make_classification as mc;
from sklearn.datasets import make_regression as mr
from sklearn.datasets import load_iris;
from sklearn.datasets import load_diabetes

from sklearn.metrics import precision_recall_curve as prc;
from sklearn.metrics import roc_curve as roc;
from sklearn.metrics import classification_report as report;
from sklearn.metrics import roc_auc_score;
from sklearn.metrics import precision_score;
from sklearn.metrics import recall_score;
from sklearn.metrics import get_scorer_names
from sklearn.metrics import f1_score as f1
from sklearn.metrics import accuracy_score as accuracy
from sklearn.metrics import r2_score

from sklearn.linear_model import LogisticRegression as LR;
from sklearn.linear_model import LinearRegression as LinReg;
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import SGDClassifier as SGDC;

from sklearn.model_selection import train_test_split;
from sklearn.model_selection import cross_validate;
from sklearn.model_selection import KFold;
from sklearn.model_selection import LearningCurveDisplay
from sklearn.model_selection import RandomizedSearchCV as RSCV;
from sklearn.model_selection import LeavePOut;
from sklearn.model_selection import GridSearchCV as GSCV;

from sklearn.neighbors import KNeighborsRegressor as KNNR
from sklearn.neighbors import KNeighborsClassifier as KNNC

from sklearn.gaussian_process import GaussianProcessClassifier as GPC
from sklearn.gaussian_process.kernels import RBF

from sklearn.preprocessing import OneHotEncoder as OHE;
from sklearn.preprocessing import StandardScaler as SC;

from sklearn.ensemble import RandomForestClassifier as RFC;
from sklearn.ensemble import RandomForestRegressor as RFR;
from sklearn.ensemble import GradientBoostingClassifier as GBC
from sklearn.ensemble import ExtraTreesClassifier as ETC;
from sklearn.ensemble import ExtraTreesRegressor as ETR;
from sklearn.ensemble import AdaBoostClassifier as AdaBoost;
from sklearn.ensemble import AdaBoostRegressor;
from sklearn.ensemble import GradientBoostingRegressor as GBR;

from sklearn.tree import DecisionTreeClassifier as DTC;
from sklearn.tree import DecisionTreeRegressor  as DTR

from sklearn.naive_bayes import GaussianNB as GNB;

from sklearn.svm import SVC;
from sklearn.svm import SVR;

from sklearn.neural_network import MLPClassifier as MLPC
from sklearn.neural_network import MLPRegressor as MLPR;

from yellowbrick.classifier import ROCAUC;
from yellowbrick.classifier import PrecisionRecallCurve;

from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA;

from yellowbrick.model_selection import LearningCurve

from sklearn.pipeline import make_pipeline;

from tqdm.notebook import tqdm

import warnings;warnings.simplefilter("ignore")

#1 При выборе модели мы обычно выбираем ту, которая дает наибольшее значение какой-то метрики. Но это означает, что эта оценка метрики уже становится завышенной. Поэтому для объективной оценки качества модели ее опять следует оценить на новом наборе данных. Поэтому для выбора модели нужно разделить выборку на три части - обучающую, валидационную и тестовую. Повторите первую часть работы, но с таким разбиением и получите несмещенную оценку качества наилучшей модели.

In [ ]:
X, Y = mc(n_samples=1000, n_classes=2, n_features=5, n_redundant=0, random_state=1)

df_splitter = pd.DataFrame(np.hstack([X,Y.reshape([-1,1])]))

ind = np.random.randint(0,len(df_splitter)-1,len(df_splitter)//5)
trains = df_splitter.drop(ind)
tests  = df_splitter.loc[ind]


X_train, X_test, Y_train, Y_test = train_test_split(trains.iloc[:,:-1], trains.iloc[:,-1], test_size=0.2)

names = ["Nearest_Neighbors", "Linear_SVM", "Polynomial_SVM", "RBF_SVM", "Gaussian_Process",
         "Gradient_Boosting", "Decision_Tree", "Extra_Trees", "Random_Forest", "Neural_Net", "AdaBoost",
         "Naive_Bayes", "QDA", "SGD"]

classifiers = [
    KNNC(3),
    SVC(kernel="linear", C=0.025),
    SVC(kernel="poly", degree=3, C=0.025),
    SVC(kernel="rbf", C=1, gamma=2),
    GPC(1.0 * RBF(1.0)),
    GBC(n_estimators=100, learning_rate=1.0),
    DTC(max_depth=5),
    ETC(n_estimators=10, min_samples_split=2),
    RFC(max_depth=5, n_estimators=100),
    MLPC(alpha=1, max_iter=1000),
    AdaBoost(n_estimators=100),
    GNB(),
    QDA(),
    SGDC(loss="hinge", penalty="l2")]

scores = []
for name, clf in zip(names, classifiers):
    clf.fit(X_train, Y_train)
    score = clf.score(X_test, Y_test)
    scores.append(score)

df = pd.DataFrame()
df['name'] = names
df['score'] = scores
df.sort_values(by=["score"], ascending=False, inplace=True)
#df


cm = sns.light_palette("green", as_cmap=True)
s = df.style.background_gradient(cmap=cm)
display(s)


scores = []
for name, clf in zip(names, classifiers):
    clf.fit(X_train, Y_train)
    score = clf.score(tests.iloc[:,:-1], tests.iloc[:,-1])
    scores.append(score)

df = pd.DataFrame()
df['name'] = names
df['score'] = scores
df.sort_values(by=["score"], ascending=False, inplace=True)
df


cm = sns.light_palette("green", as_cmap=True)
s = df.style.background_gradient(cmap=cm)
display(s)

,name,score
9,Neural_Net,0.884146
4,Gaussian_Process,0.878049
2,Polynomial_SVM,0.871951
12,QDA,0.871951
13,SGD,0.871951
1,Linear_SVM,0.871951
11,Naive_Bayes,0.871951
8,Random_Forest,0.865854
10,AdaBoost,0.859756
0,Nearest_Neighbors,0.847561


,name,score
6,Decision_Tree,0.885000
1,Linear_SVM,0.880000
8,Random_Forest,0.880000
9,Neural_Net,0.880000
4,Gaussian_Process,0.880000
12,QDA,0.875000
11,Naive_Bayes,0.870000
10,AdaBoost,0.870000
7,Extra_Trees,0.860000
5,Gradient_Boosting,0.860000


#2 Оптимизируйте гиперпараметры той же модели, но другим методом - случайным поиском. Сравните полученные результаты.

In [ ]:
#Decision_Tree
X, Y = mc(n_samples=1000, n_classes=2, n_features=5, n_redundant=0, random_state=1)
X_train, X_test, Y_train, Y_test = train_test_split(trains.iloc[:,:-1], trains.iloc[:,-1], test_size=0.2)
param_dist = {
    'max_depth': np.random.randint(3, 60,40),        # глубина дерева
    'min_samples_split': np.random.randint(2, 40,10),
    'max_features': ['sqrt', 'log2'],  # категориальный параметр
    'splitter':['best', 'random'],
    'criterion': ['gini', 'entropy', "log_loss"],
}

m = DTC()
SEARCH = RSCV(m,param_dist,n_iter=50,cv=5)
SEARCH.fit(X_train,Y_train)
display(SEARCH.best_params_)
SEARCH.score(X_test,Y_test)

{'splitter': 'random',
 'min_samples_split': np.int64(28),
 'max_features': 'sqrt',
 'max_depth': np.int64(10),
 'criterion': 'log_loss'}

0.8719512195121951

#3 Во второй задаче найдите наиболее эффективный класс моделей, а затем оптимизируйте гиперпараметры у этой модели. Сравните, насколько лучше получилась модель.

https://github.com/nguyen-toan/ISLR/blob/master/dataset/Advertising.csv

In [ ]:
p = pd.read_csv('/content/Advertising.csv').iloc[:,1:]
X = p.drop('sales',axis=1)
#X = SC().fit_transform(X)
X = pd.DataFrame(SC().fit_transform(X),columns=X.columns)
Y = p[['sales']]
print(X.shape)
print(Y.values.reshape([-1,1]).shape)
#df_splitter = pd.DataFrame(np.stack([X,Y.values.reshape([-1,1])]),columns = p.columns)
df_splitter = X.join(Y)
#X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

ind = np.random.randint(0,len(df_splitter)-1,len(df_splitter)//5)
trains = df_splitter.drop(ind)
tests  = df_splitter.loc[ind]

X_train, X_test, Y_train, Y_test = train_test_split(trains.iloc[:,:-1], trains.iloc[:,-1], test_size=0.2)

names = [
    "Linear_Regression",
    "Lasso",
    "ElasticNet",
    "KNN_Regression",
    "Decision_Tree_Regression",
    "Linear_SVR",
    "Polynomial_SVR",
    "RBF_SVR",
    "Gradient_Boosting_Regression",
    "Random_Forest_Regression",
    "Extra_Trees_Regression",
    "MLP_Regression",
    "AdaBoost_Regression"
]

regressors = [
    LinReg(),                          # Линейная регрессия
    Lasso(),                           # L1-регуляризация
    ElasticNet(),                      # ElasticNet (L1 + L2)
    KNNR(),                            # K-ближайших соседей (регрессия)
    DTR(),                             # Дерево решений
    SVR(kernel="linear"),              # SVM с линейным ядром
    SVR(kernel="poly", degree=3),      # SVM с полиномиальным ядром
    SVR(kernel="rbf"),                 # SVM с RBF-ядром
    GBR(n_estimators=100),             # Градиентный бустинг (можно использовать loss='ls' для MSE)
    RFR(n_estimators=100),             # Случайный лес
    ETR(n_estimators=100),             # Extra Trees
    MLPR(alpha=1, max_iter=1000),      # Нейронная сеть
    AdaBoostRegressor(n_estimators=100)         # AdaBoost (базовый оценщик по умолчанию — DecisionTreeRegressor)
]

scores = []
for name, clf in zip(names, regressors):
    print(name)
    clf.fit(X_train, Y_train)
    score = clf.score(X_test, Y_test)
    scores.append(score)

df = pd.DataFrame()
df['name'] = names
df['score'] = scores
df.sort_values(by=["score"], ascending=False, inplace=True)
#df


cm = sns.light_palette("green", as_cmap=True)
s = df.style.background_gradient(cmap=cm)
display(s)

scores = []
for name, clf in zip(names, regressors):
    clf.fit(X_train, Y_train)
    score = clf.score(tests.iloc[:,:-1], tests.iloc[:,-1])
    scores.append(score)

df = pd.DataFrame()
df['name'] = names
df['score'] = scores
df.sort_values(by=["score"], ascending=False, inplace=True)
df


cm = sns.light_palette("green", as_cmap=True)
s = df.style.background_gradient(cmap=cm)
display(s)

(200, 3)
(200, 1)
Linear_Regression
Lasso
ElasticNet
KNN_Regression
Decision_Tree_Regression
Linear_SVR
Polynomial_SVR
RBF_SVR
Gradient_Boosting_Regression
Random_Forest_Regression
Extra_Trees_Regression
MLP_Regression
AdaBoost_Regression


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


,name,score
10,Extra_Trees_Regression,0.994916
9,Random_Forest_Regression,0.984098
8,Gradient_Boosting_Regression,0.983014
12,AdaBoost_Regression,0.969794
7,RBF_SVR,0.966147
3,KNN_Regression,0.961630
4,Decision_Tree_Regression,0.947905
5,Linear_SVR,0.913492
11,MLP_Regression,0.909782
0,Linear_Regression,0.884409


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


,name,score
10,Extra_Trees_Regression,0.991552
9,Random_Forest_Regression,0.984498
8,Gradient_Boosting_Regression,0.983532
12,AdaBoost_Regression,0.966381
4,Decision_Tree_Regression,0.951471
3,KNN_Regression,0.935393
11,MLP_Regression,0.918454
7,RBF_SVR,0.901236
0,Linear_Regression,0.859036
5,Linear_SVR,0.844089


In [ ]:
#ETR
m = ETR()
'''
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2', 0.5, 0.8],
    'bootstrap': [True, False],
    'criterion': ['squared_error', 'absolute_error', 'friedman_mse']
}
'''
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_features': [0.8, 1.0],  # Обязательно проверить 1.0
    'max_depth': [None, 10, 20], # None должно быть в списке
    'min_samples_split': [2, 5],
    'bootstrap': [True, False]   # Добавить этот параметр
}

grid_model = GSCV(
    estimator=m,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=2,
    verbose=1)



p = pd.read_csv('/content/Advertising.csv').iloc[:,1:]
X = p.drop('sales',axis=1)
X = pd.DataFrame(SC().fit_transform(X),columns=X.columns)
Y = p[['sales']]


X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

grid_model.fit(X_train,Y_train)
print(r2_score(Y_test, grid_model.predict(X_test)))

Fitting 2 folds for each of 72 candidates, totalling 144 fits
0.981151809491476
